# LiDAR/Wave v0.6.0 Test Harness — upload A + B, then run

This notebook is the **test-harness** version of the sweep lab.

**A = engine file**: `lidar_lenses_wave_v060.py` or newer  
**B = workbook**: `lidar_wave_test_plan_v060.xlsx` or compatible

It does four jobs:

1. Run preset sweeps with classifier-threshold columns.
2. Generate per-material raw channel reports.
3. Run deterministic repeat tests.
4. Run edge-case tests.

The goal is not just to chase the highest score. The goal is to document behavior and catch regressions.


In [ ]:
# 1) Upload A + B and import the engine directly from the uploaded .py file.
from pathlib import Path
import sys, os, json, zipfile, shutil, importlib.util, hashlib, time, math
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

WORK_DIR = Path("/content/lidar_v060_test_harness")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

print("Working folder:", WORK_DIR)
print("\nUpload two files:")
print("  A) lidar_lenses_wave_v060.py  (or newer engine .py)")
print("  B) lidar_wave_test_plan_v060.xlsx  (or your edited test plan)")
print()

try:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        path = WORK_DIR / name
        path.write_bytes(data)
except Exception as e:
    print("Colab upload widget was not available or was skipped.")
    print("Put the .py and .xlsx files in:", WORK_DIR)
    print("Details:", repr(e))

py_files = sorted(WORK_DIR.glob("*.py"), key=lambda p: p.stat().st_mtime, reverse=True)
xlsx_files = sorted(WORK_DIR.glob("*.xlsx"), key=lambda p: p.stat().st_mtime, reverse=True)

if not py_files:
    raise FileNotFoundError(f"No engine .py file found in {WORK_DIR}. Upload lidar_lenses_wave_v060.py or newer.")

engine_path = py_files[0]
plan_path = xlsx_files[0] if xlsx_files else None

print("Engine file:", engine_path.name)
print("Plan file:", plan_path.name if plan_path else "(will create default plan)")

spec = importlib.util.spec_from_file_location("llw_uploaded_engine", str(engine_path))
llw = importlib.util.module_from_spec(spec)
spec.loader.exec_module(llw)

doc = (llw.__doc__ or "").strip().splitlines()
print("\nLoaded engine from:", engine_path)
if doc:
    print("Engine doc:", doc[0])

OUT_DIR = WORK_DIR / "test_outputs_v060"
OUT_DIR.mkdir(exist_ok=True)
print("Output folder:", OUT_DIR)


## Scene builders

Includes the four main stress scenes plus edge-case scenes.


In [ ]:
# 2) Quick procedural scenes.
def _R(rx=0, ry=0, rz=0):
    return llw.make_rotation_matrix(rx, ry, rz)

def _add(prims, shape, center, half_extents, color, piece_type, rotation=None, transparency=0.0, piece_id=None):
    if rotation is None:
        rotation = _R()
    if piece_id is None:
        piece_id = len(prims)
    prims.append(llw.Primitive(
        shape=shape,
        center=np.array(center, dtype=np.float64),
        half_extents=np.array(half_extents, dtype=np.float64),
        rotation_matrix=rotation.T,
        inv_rotation_matrix=rotation,
        color=tuple(color),
        piece_id=int(piece_id),
        piece_type=str(piece_type),
        transparency=float(transparency),
    ))

def build_mini_saloon():
    prims = []
    # floor and walls
    _add(prims, "box", [0, -0.05, 0], [7.0, 0.05, 6.0], (0.45, 0.31, 0.18), "wood_floor")
    _add(prims, "box", [0, 1.6, 3.05], [7.0, 1.6, 0.12], (0.50, 0.32, 0.18), "wood_wall")
    _add(prims, "box", [-3.55, 1.6, 0], [0.12, 1.6, 6.0], (0.48, 0.28, 0.16), "wood_wall")
    _add(prims, "box", [3.55, 1.6, 0], [0.12, 1.6, 6.0], (0.48, 0.28, 0.16), "wood_wall")
    # bar counter and back shelf
    _add(prims, "box", [0.0, 0.65, 2.0], [4.6, 0.65, 0.45], (0.35, 0.20, 0.10), "bar_counter")
    _add(prims, "box", [0.0, 1.65, 2.85], [4.8, 0.08, 0.15], (0.30, 0.18, 0.09), "shelf")
    for x in np.linspace(-2.0, 2.0, 7):
        _add(prims, "cylinder", [x, 1.95, 2.67], [0.06, 0.22, 0.0], (0.1, 0.55, 0.35), "bottle", rotation=_R())
    # tables/chairs
    for cx, cz in [(-1.8, -0.6), (1.5, -0.7), (0.0, -2.1)]:
        _add(prims, "cylinder", [cx, 0.52, cz], [0.45, 0.06, 0.0], (0.34, 0.18, 0.08), "table_top")
        _add(prims, "cylinder", [cx, 0.25, cz], [0.08, 0.25, 0.0], (0.25, 0.14, 0.06), "table_leg")
        for dx, dz, rot in [(0.75,0,0),(-0.75,0,0),(0,0.75,90),(0,-0.75,90)]:
            _add(prims, "box", [cx+dx, 0.28, cz+dz], [0.32,0.28,0.32], (0.38,0.20,0.08), "chair")
            _add(prims, "box", [cx+dx, 0.72, cz+dz], [0.35,0.55,0.08], (0.30,0.15,0.06), "chair_back", rotation=_R(0,rot,0))
    # stools
    for x in np.linspace(-2.0, 2.0, 5):
        _add(prims, "cylinder", [x, 0.48, 1.25], [0.18, 0.07, 0.0], (0.30,0.14,0.06), "stool")
        _add(prims, "cylinder", [x, 0.24, 1.25], [0.05, 0.24, 0.0], (0.18,0.10,0.05), "stool_leg")
    return prims

def build_occluder_gate():
    prims = []
    _add(prims, "box", [0, -0.05, 0], [9, 0.05, 7], (0.32,0.30,0.22), "ground")
    _add(prims, "box", [0, 1.2, 2.8], [7.5, 1.2, 0.12], (0.55,0.55,0.52), "back_wall")
    # fence/gate
    for x in np.linspace(-3, 3, 9):
        _add(prims, "cylinder", [x, 0.8, 0.2], [0.045, 0.8, 0.0], (0.35,0.22,0.10), "fence", transparency=0.55)
    for y in [0.55, 1.15]:
        _add(prims, "box", [0, y, 0.2], [6.4, 0.04, 0.06], (0.32,0.20,0.09), "fence", transparency=0.55)
    # foliage blobs
    for x,z,r in [(-2.8,1.2,0.9),(-1.5,1.5,0.75),(1.8,1.3,0.85),(3.0,1.0,0.7)]:
        _add(prims, "sphere", [x, 1.4, z], [r,r,r], (0.16,0.50,0.18), "canopy", transparency=0.65)
        _add(prims, "cylinder", [x, 0.55, z], [0.10,0.55,0.0], (0.28,0.16,0.08), "trunk")
    # hard boxes behind occluder
    for x in [-1.2, 1.2]:
        _add(prims, "box", [x, 0.5, 2.0], [0.7,0.5,0.5], (0.65,0.62,0.55), "stone")
    return prims

def build_material_targets():
    prims = []
    _add(prims, "box", [0, -0.05, 0], [9, 0.05, 5], (0.30,0.30,0.28), "ground")
    mats = [
        ("wood",   (-3.2,0.6,0), (0.45,0.25,0.12), "cylinder"),
        ("metal",  (-1.9,0.6,0), (0.72,0.74,0.76), "box"),
        ("glass",  (-0.6,0.6,0), (0.60,0.80,0.95), "box"),
        ("cloth",  (0.7,0.6,0),  (0.65,0.30,0.25), "box"),
        ("stone",  (2.0,0.6,0),  (0.45,0.45,0.46), "sphere"),
        ("foliage",(3.3,0.75,0), (0.18,0.52,0.20), "sphere"),
    ]
    for name, center, color, shape in mats:
        if shape == "sphere":
            _add(prims, shape, center, [0.55,0.55,0.55], color, name, transparency=0.50 if name=="foliage" else 0.0)
        elif shape == "cylinder":
            _add(prims, shape, center, [0.35,0.6,0.0], color, name)
        else:
            _add(prims, shape, center, [0.9,1.0,0.7], color, name)
    return prims

def build_scene(scene_name):
    s = str(scene_name).strip()
    if s == "cabin_demo":
        return llw._build_demo_scene()
    if s == "mini_saloon":
        return build_mini_saloon()
    if s == "occluder_gate":
        return build_occluder_gate()
    if s == "material_targets":
        return build_material_targets()
    if s == "one_box":
        prims = []
        _add(prims, "box", [0, 0.5, 0], [1.0, 1.0, 1.0], (0.6,0.5,0.4), "box")
        return prims
    if s == "transparent_sphere":
        prims = []
        _add(prims, "box", [0, -0.05, 0], [5.0, 0.05, 5.0], (0.3,0.3,0.25), "ground")
        _add(prims, "sphere", [0, 1.0, 0], [0.8,0.8,0.8], (0.2,0.55,0.25), "foliage", transparency=0.75)
        _add(prims, "box", [0, 0.5, 1.8], [1.0, 0.5, 0.3], (0.7,0.7,0.65), "stone")
        return prims
    if s == "high_transparency":
        prims = []
        _add(prims, "box", [0, -0.05, 0], [5.0, 0.05, 5.0], (0.3,0.3,0.25), "ground")
        for x in np.linspace(-1.5, 1.5, 5):
            _add(prims, "sphere", [x, 1.0, 0], [0.7,0.7,0.7], (0.2,0.55,0.25), "foliage", transparency=0.99)
        return prims
    if s == "flat_topdown":
        prims = []
        _add(prims, "box", [0, -0.05, 0], [8.0, 0.05, 8.0], (0.35,0.32,0.25), "ground")
        for x,z in [(-2,-2),(2,-2),(-2,2),(2,2)]:
            _add(prims, "box", [x, 0.15, z], [0.8,0.3,0.8], (0.5,0.25,0.12), "low_box")
        return prims
    if s == "empty_scene":
        return []
    raise ValueError(f"Unknown scene {scene_name!r}. Valid: cabin_demo, mini_saloon, occluder_gate, material_targets, one_box, transparent_sphere, high_transparency, flat_topdown, empty_scene")

print("Scene builders ready.")


def maybe_apply_material_priors(prims, use_material_priors=True):
    if use_material_priors and hasattr(llw, "apply_material_prior_transparency"):
        llw.apply_material_prior_transparency(prims)
    return prims

print("v0.6.0 scene builders ready.")


## Load workbook

Reads `SweepPlan`, `MaterialDiagnostics`, `DeterminismPlan`, and `EdgeCases` if present.


In [ ]:
# 3) Load workbook sheets.
def read_sheet_or_empty(path, sheet_name):
    if path is None:
        return pd.DataFrame()
    try:
        xl = pd.ExcelFile(path)
        if sheet_name not in xl.sheet_names:
            return pd.DataFrame()
        df = pd.read_excel(path, sheet_name=sheet_name)
        df.columns = [str(c).strip() for c in df.columns]
        if "enabled" in df.columns:
            # treat blank enabled as True
            enabled = df["enabled"].apply(lambda x: True if pd.isna(x) else str(x).strip().lower() not in ("0","false","no","off"))
            df = df[enabled].copy()
        return df
    except Exception as e:
        print(f"Could not read sheet {sheet_name!r}:", repr(e))
        return pd.DataFrame()

DEFAULT_SWEEP = pd.DataFrame([
    dict(enabled=True, scene="mini_saloon", preset="indoor_structure", width=480, height=320, rays_per_pixel=8, stack=4, carrier_strength=0.50, edge_anti_min=0.38, adaptive_edge_percentile=94, carrier_mode="ensemble", include_polarization=False, seed=42, notes="indoor regression"),
    dict(enabled=True, scene="occluder_gate", preset="outdoor_occlusion", width=480, height=320, rays_per_pixel=8, stack=4, carrier_strength=0.50, edge_anti_min=0.34, adaptive_edge_percentile=92, carrier_mode="ensemble", include_polarization=False, seed=42, notes="occlusion regression"),
    dict(enabled=True, scene="material_targets", preset="material_scan", width=480, height=320, rays_per_pixel=8, stack=4, carrier_strength=0.30, edge_anti_min=0.30, adaptive_edge_percentile=94, carrier_mode="ensemble", include_polarization=True, use_material_priors=True, seed=42, notes="material baseline"),
    dict(enabled=True, scene="material_targets", preset="material_scan", width=480, height=320, rays_per_pixel=8, stack=4, carrier_strength=0.30, edge_anti_min=0.30, adaptive_edge_percentile=94, carrier_mode="ensemble", include_polarization=True, use_material_priors=True, soft_material_texture_min=0.001, seed=42, notes="wiring test: soft threshold should change labels"),
])

sweep_df = read_sheet_or_empty(plan_path, "SweepPlan")
material_diag_df = read_sheet_or_empty(plan_path, "MaterialDiagnostics")
determinism_df = read_sheet_or_empty(plan_path, "DeterminismPlan")
edge_df = read_sheet_or_empty(plan_path, "EdgeCases")

if sweep_df.empty:
    sweep_df = DEFAULT_SWEEP.copy()

print("Sweep rows:", len(sweep_df))
print("Material diagnostic rows:", len(material_diag_df))
print("Determinism rows:", len(determinism_df))
print("Edge-case rows:", len(edge_df))

display(sweep_df.head(20))


## Shared runner

The runner forwards classifier threshold columns into the engine when present.


In [ ]:
# 4) Shared runner utilities.
def _clean_value(v):
    if pd.isna(v):
        return None
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    return v

BASE_OVERRIDE_COLUMNS = [
    "width", "height", "rays_per_pixel", "stack", "carrier_strength",
    "edge_anti_min", "adaptive_edge_percentile", "carrier_mode",
    "include_polarization", "attenuation_per_meter", "depth_wavelength",
    "sampling_mode", "auto_frame", "fov_deg", "camera"
]

CLASSIFIER_COLUMNS = list(getattr(llw, "CLASSIFIER_KWARG_NAMES", []))
OVERRIDE_COLUMNS = sorted(set(BASE_OVERRIDE_COLUMNS + CLASSIFIER_COLUMNS))

def row_bool(v, default=False):
    if v is None or pd.isna(v):
        return default
    if isinstance(v, str):
        return v.strip().lower() in ("1", "true", "yes", "y", "on")
    return bool(v)

def build_overrides(row):
    overrides = {}
    for col in OVERRIDE_COLUMNS:
        if col in row.index:
            val = _clean_value(row[col])
            if val is None:
                continue
            if col in ["width", "height", "rays_per_pixel", "stack"]:
                val = int(val)
            if col in ["include_polarization", "auto_frame"]:
                val = row_bool(val)
            overrides[col] = val
    return overrides

def channel_stats(channels):
    out = {}
    if not channels:
        return out
    hits = channels.get("hit_count")
    mask = hits > 0 if hits is not None else None
    for name in ["light_anti", "sound_anti", "light_coh", "sound_coh",
                 "acoustic_intensity", "ultrasonic_intensity", "acoustic_texture",
                 "polarization", "depth_variance"]:
        if name in channels:
            arr = np.asarray(channels[name])
            vals = arr[mask] if mask is not None and mask.any() else arr.ravel()
            vals = vals[np.isfinite(vals)]
            if vals.size:
                out[f"{name}_p50"] = float(np.percentile(vals, 50))
                out[f"{name}_p90"] = float(np.percentile(vals, 90))
                out[f"{name}_p95"] = float(np.percentile(vals, 95))
                out[f"{name}_mean"] = float(vals.mean())
    return out

def label_fractions(counts):
    counts = counts or {}
    hit_px = sum(v for k, v in counts.items() if k != "sky")
    out = {}
    for name in ["solid_surface", "geom_edge", "partial_occluder", "foliage",
                 "hard_smooth", "soft_material", "optical_only", "acoustic_only", "uncertain"]:
        out[f"{name}_frac"] = float(counts.get(name, 0) / max(hit_px, 1))
    out["hit_labeled_pixels"] = int(hit_px)
    return out

def heuristic_score(row):
    cov = float(row.get("coverage", 0.0) or 0.0)
    span = float(row.get("depth_span_p05_p95", 0.0) or 0.0)
    la = float(row.get("light_anti_p95", 0.0) or 0.0)
    sa = float(row.get("sound_anti_p95", 0.0) or 0.0)
    edge = float(row.get("geom_edge_frac", 0.0) or 0.0)
    partial = float(row.get("partial_occluder_frac", 0.0) or 0.0)
    # Balanced score for ranking top visual sheets; not a scientific objective.
    return (
        0.30 * min(cov / 0.75, 1.0) +
        0.20 * min(span / 8.0, 1.0) +
        0.25 * min((la + sa) / 1.4, 1.0) +
        0.15 * max(0.0, 1.0 - abs(edge - 0.08) / 0.20) +
        0.10 * min(partial / 0.08, 1.0)
    )

def run_preset_row(row, run_index, root_dir):
    scene_name = str(row["scene"]).strip()
    preset_name = str(row["preset"]).strip()
    seed = int(row.get("seed", 42) if not pd.isna(row.get("seed", 42)) else 42)
    use_priors = row_bool(row.get("use_material_priors", False), default=False)

    overrides = build_overrides(row)
    prims = build_scene(scene_name)
    maybe_apply_material_priors(prims, use_priors)

    run_name = f"{run_index:03d}_{scene_name}_{preset_name}"
    run_dir = root_dir / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    kwargs = dict(
        prims=prims,
        preset_name=preset_name,
        scene_name=scene_name,
        out_dir=str(run_dir),
        seed=seed,
        **overrides,
    )
    if use_priors and hasattr(llw, "material_prior_acoustic"):
        kwargs.update(
            impedance_fn=llw.material_prior_acoustic,
            impedance_ult_fn=llw.material_prior_ultrasonic,
            polarization_fn=llw.material_prior_polarization,
        )

    t0 = time.time()
    result = llw.run_sensor_preset(**kwargs)
    elapsed = time.time() - t0

    diag = result["diagnostics"]
    stats = diag.get("depth_stats", {})
    counts = diag.get("classification_counts") or {}
    channels = result.get("channels")

    row_out = {
        "run_index": run_index,
        "scene": scene_name,
        "preset": preset_name,
        "seed": seed,
        "notes": row.get("notes", ""),
        "use_material_priors": use_priors,
        "runtime_seconds": round(float(elapsed), 3),
        "coverage": float(stats.get("coverage", 0.0) or 0.0),
        "depth_span_p05_p95": float(stats.get("depth_span_p05_p95", 0.0) or 0.0),
        "depth_p50": stats.get("depth_p50"),
        "contact_sheet": result.get("paths", {}).get("contact_sheet"),
        "diagnostics_path": result.get("paths", {}).get("diagnostics"),
        "channels_path": result.get("paths", {}).get("channels"),
    }
    for col in OVERRIDE_COLUMNS:
        if col in row.index and not pd.isna(row[col]):
            row_out[col] = row[col]

    row_out.update(channel_stats(channels))
    row_out.update(label_fractions(counts))
    row_out["score"] = heuristic_score(row_out)

    # Save per-run material report when available.
    report = result.get("material_report") or []
    if report:
        report_path = run_dir / "material_channel_report.csv"
        pd.DataFrame(report).to_csv(report_path, index=False)
        row_out["material_report_path"] = str(report_path)
    return row_out, result


## Main sweep


In [ ]:
# 5) Run main sweep.
sweep_rows = []
sweep_results = []
for i, (_, row) in enumerate(sweep_df.iterrows(), start=1):
    try:
        print(f"[sweep {i}/{len(sweep_df)}] {row['scene']} / {row['preset']}")
        row_out, result = run_preset_row(row, i, OUT_DIR / "sweep_runs")
        sweep_rows.append(row_out)
        sweep_results.append(result)
    except Exception as e:
        print("  ERROR:", repr(e))
        sweep_rows.append({
            "run_index": i,
            "scene": row.get("scene", ""),
            "preset": row.get("preset", ""),
            "error": repr(e),
            "score": 0.0,
        })

metrics_df = pd.DataFrame(sweep_rows)
metrics_df = metrics_df.sort_values("score", ascending=False, na_position="last")
metrics_csv = OUT_DIR / "sweep_metrics.csv"
metrics_xlsx = OUT_DIR / "sweep_metrics.xlsx"
metrics_df.to_csv(metrics_csv, index=False)
metrics_df.to_excel(metrics_xlsx, index=False)
print("Saved:", metrics_csv)
print("Saved:", metrics_xlsx)
display(metrics_df.head(20))


## Material diagnostics

This is the most important section for material tuning.


In [ ]:
# 6) Material channel diagnostics across plan rows.
# This is the key report for deciding whether material labels fail because
# thresholds are wrong or because raw channels overlap.
material_report_rows = []

if material_diag_df.empty:
    print("No MaterialDiagnostics sheet found/enabled; skipping.")
else:
    for i, (_, row) in enumerate(material_diag_df.iterrows(), start=1):
        try:
            scene_name = str(row.get("scene", "material_targets")).strip()
            preset_name = str(row.get("preset", "material_scan")).strip()
            seed = int(row.get("seed", 42) if not pd.isna(row.get("seed", 42)) else 42)
            use_priors = row_bool(row.get("use_material_priors", True), default=True)
            overrides = build_overrides(row)
            prims = build_scene(scene_name)
            maybe_apply_material_priors(prims, use_priors)

            run_dir = OUT_DIR / "material_diagnostics" / f"{i:03d}_{scene_name}_{preset_name}"
            run_dir.mkdir(parents=True, exist_ok=True)

            kwargs = dict(prims=prims, preset_name=preset_name, scene_name=scene_name,
                          out_dir=str(run_dir), seed=seed, **overrides)
            if use_priors and hasattr(llw, "material_prior_acoustic"):
                kwargs.update(
                    impedance_fn=llw.material_prior_acoustic,
                    impedance_ult_fn=llw.material_prior_ultrasonic,
                    polarization_fn=llw.material_prior_polarization,
                )

            result = llw.run_sensor_preset(**kwargs)
            report = result.get("material_report") or []
            for r in report:
                r.update({
                    "diag_index": i,
                    "scene": scene_name,
                    "preset": preset_name,
                    "view_note": row.get("view_note", ""),
                    "seed": seed,
                    "use_material_priors": use_priors,
                })
                material_report_rows.append(r)
            print(f"[material {i}] {scene_name}/{preset_name}: {len(report)} material rows")
        except Exception as e:
            print(f"[material {i}] ERROR:", repr(e))
            material_report_rows.append({"diag_index": i, "error": repr(e)})

material_report_df = pd.DataFrame(material_report_rows)
if not material_report_df.empty:
    material_csv = OUT_DIR / "material_channel_report.csv"
    material_xlsx = OUT_DIR / "material_channel_report.xlsx"
    material_report_df.to_csv(material_csv, index=False)
    material_report_df.to_excel(material_xlsx, index=False)
    print("Saved:", material_csv)
    print("Saved:", material_xlsx)
    display(material_report_df.head(30))

    # Compact discrimination summary by channel.
    summary_rows = []
    for ch in ["acoustic_intensity_mean", "ultrasonic_intensity_mean", "acoustic_texture_mean", "polarization_mean", "depth_variance_mean"]:
        if ch in material_report_df.columns:
            vals = material_report_df.groupby("piece_type")[ch].mean().dropna()
            if len(vals):
                summary_rows.append({
                    "channel": ch,
                    "n_materials": int(len(vals)),
                    "min": float(vals.min()),
                    "max": float(vals.max()),
                    "spread": float(vals.max() - vals.min()),
                    "materials_sorted": "; ".join(f"{k}={v:.3f}" for k, v in vals.sort_values().items())
                })
    discrim_df = pd.DataFrame(summary_rows)
    discrim_path = OUT_DIR / "material_discrimination_summary.csv"
    discrim_df.to_csv(discrim_path, index=False)
    print("Saved:", discrim_path)
    display(discrim_df)
else:
    print("No material report rows.")


## Determinism tests


In [ ]:
# 7) Determinism / seed-variance tests.
det_rows = []

if determinism_df.empty:
    print("No DeterminismPlan sheet found/enabled; skipping.")
else:
    for i, (_, row) in enumerate(determinism_df.iterrows(), start=1):
        repeats = int(row.get("repeats", 3) if not pd.isna(row.get("repeats", 3)) else 3)
        mode = str(row.get("mode", "same_seed")).strip()
        base_seed = int(row.get("seed", 42) if not pd.isna(row.get("seed", 42)) else 42)
        metrics = []
        hashes = []
        for rep in range(repeats):
            r = row.copy()
            if mode == "different_seed":
                r["seed"] = base_seed + rep
            else:
                r["seed"] = base_seed
            try:
                row_out, result = run_preset_row(r, rep + 1, OUT_DIR / "determinism_runs" / f"case_{i:03d}")
                metrics.append(row_out)
                p = row_out.get("contact_sheet")
                if p and Path(p).exists():
                    hashes.append(hashlib.sha256(Path(p).read_bytes()).hexdigest())
                else:
                    hashes.append(None)
            except Exception as e:
                metrics.append({"error": repr(e)})
                hashes.append(None)
        mdf = pd.DataFrame(metrics)
        record = {
            "case_index": i,
            "scene": row.get("scene", ""),
            "preset": row.get("preset", ""),
            "mode": mode,
            "repeats": repeats,
            "unique_contact_hashes": len(set(hashes)),
            "all_hashes_equal": len(set(hashes)) == 1,
            "coverage_std": float(mdf["coverage"].std()) if "coverage" in mdf else None,
            "score_std": float(mdf["score"].std()) if "score" in mdf else None,
            "light_anti_p95_std": float(mdf["light_anti_p95"].std()) if "light_anti_p95" in mdf else None,
            "errors": int(mdf.get("error", pd.Series(dtype=object)).notna().sum()) if "error" in mdf else 0,
        }
        det_rows.append(record)
        print(f"[det {i}] {record}")

det_df = pd.DataFrame(det_rows)
if not det_df.empty:
    det_csv = OUT_DIR / "determinism_report.csv"
    det_df.to_csv(det_csv, index=False)
    print("Saved:", det_csv)
    display(det_df)


## Edge-case tests


In [ ]:
# 8) Edge-case tests.
edge_rows = []

if edge_df.empty:
    print("No EdgeCases sheet found/enabled; skipping.")
else:
    for i, (_, row) in enumerate(edge_df.iterrows(), start=1):
        test_type = str(row.get("test_type", "preset")).strip()
        scene_name = str(row.get("scene", "one_box")).strip()
        preset_name = str(row.get("preset", "full_diagnostic")).strip()
        seed = int(row.get("seed", 42) if not pd.isna(row.get("seed", 42)) else 42)
        expect_error = row_bool(row.get("expect_error", False), default=False)

        try:
            if test_type == "zero_rays":
                prims = build_scene(scene_name)
                cam = llw.make_camera_for_preset(prims, preset_name, width=64, height=48)
                burst = llw.fire_burst(cam, prims, n_samples=0, seed=seed)
                record = {
                    "case_index": i,
                    "test_type": test_type,
                    "scene": scene_name,
                    "passed": len(burst.depths) == 0,
                    "error": None,
                    "coverage": burst.coverage,
                }
            elif test_type == "zero_coverage":
                prims = build_scene(scene_name)
                # Camera looking away from the one-box scene.
                cam = llw.Camera(
                    position=np.array([0.0, 1.5, 4.0]),
                    target=np.array([0.0, 1.5, 8.0]),
                    width=160, height=100, lens="pinhole", fov_deg=45,
                    sampling_mode="halton",
                )
                burst = llw.fire_burst(cam, prims, n_samples=160*100, seed=seed)
                record = {
                    "case_index": i,
                    "test_type": test_type,
                    "scene": scene_name,
                    "passed": burst.coverage < 0.01,
                    "error": None,
                    "coverage": burst.coverage,
                }
            else:
                row_out, result = run_preset_row(row, i, OUT_DIR / "edge_case_runs")
                record = dict(row_out)
                record.update({"case_index": i, "test_type": test_type, "passed": True, "error": None})
            if expect_error and record.get("error") is None:
                record["passed"] = False
                record["error"] = "expected_error_but_succeeded"
        except Exception as e:
            record = {
                "case_index": i,
                "test_type": test_type,
                "scene": scene_name,
                "preset": preset_name,
                "passed": bool(expect_error),
                "error": repr(e),
            }
        edge_rows.append(record)
        print(f"[edge {i}] {record.get('test_type')} passed={record.get('passed')} error={record.get('error')}")

edge_report_df = pd.DataFrame(edge_rows)
if not edge_report_df.empty:
    edge_csv = OUT_DIR / "edge_case_report.csv"
    edge_report_df.to_csv(edge_csv, index=False)
    print("Saved:", edge_csv)
    display(edge_report_df)


## Package outputs


In [ ]:
# 9) Make top-contact-sheet overview and zip everything.
def make_top_overview(metrics_df, top_n=5, thumb_w=420):
    valid = metrics_df.copy()
    if "contact_sheet" not in valid:
        return None
    valid = valid[valid["contact_sheet"].notna()].sort_values("score", ascending=False).head(top_n)
    images, labels = [], []
    for _, r in valid.iterrows():
        p = Path(str(r["contact_sheet"]))
        if not p.exists():
            continue
        im = Image.open(p).convert("RGB")
        scale = thumb_w / im.width
        im = im.resize((thumb_w, max(1, int(im.height * scale))))
        label = f"#{int(r['run_index'])} {r['scene']} / {r['preset']} | score {r['score']:.3f} | cov {r.get('coverage',0):.2f}"
        images.append(im); labels.append(label)
    if not images:
        return None
    pad, label_h = 12, 28
    W = thumb_w + 2*pad
    H = pad + sum(im.height + label_h + pad for im in images)
    out = Image.new("RGB", (W, H), (10, 10, 14))
    draw = ImageDraw.Draw(out)
    y = pad
    for im, lab in zip(images, labels):
        draw.text((pad, y), lab, fill=(225, 230, 230))
        y += label_h
        out.paste(im, (pad, y))
        y += im.height + pad
    return out

overview = make_top_overview(metrics_df, top_n=5)
overview_path = OUT_DIR / "top_contact_sheets.png"
if overview:
    overview.save(overview_path)
    display(overview)
    print("Saved overview:", overview_path)
else:
    print("No overview created; no successful contact sheets found.")

# Write a one-page manifest.
manifest = {
    "engine_file": str(engine_path.name),
    "plan_file": str(plan_path.name) if plan_path else None,
    "outputs": sorted([str(p.relative_to(OUT_DIR)) for p in OUT_DIR.rglob("*") if p.is_file()])[:2000],
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

zip_path = WORK_DIR / "lidar_wave_test_outputs_v060.zip"
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path).replace(".zip",""), "zip", OUT_DIR)
print("Zipped outputs:", zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Download manually from:", zip_path)
